# Simple Regression with a Hidden Layer

## Why a Hidden Layer?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

# Data from a known non-linear function with added noise
np.random.seed(42)

n = 100
x_raw = np.linspace(-2, 2, n)

def g_func(x):
    return x**3 + 2 * x**2 - 5 * x + 2

y_raw = g_func(x_raw) + np.random.normal(0, 0.5, size=n)

# Scale the data to zero mean and unit variance (important for gradient descent)
x_scaled = (x_raw - np.mean(x_raw)) / np.std(x_raw)
y_scaled = (y_raw - np.mean(y_raw)) / np.std(y_raw)

# The network expects 2D arrays: n x 1 for the input and for the target
X = x_scaled.reshape(-1, 1)
y = y_scaled.reshape(-1, 1)
print("X shape:", X.shape, " y shape:", y.shape)

# Best straight line (the single neuron of the previous chapter)
slope, intercept = np.polyfit(x_scaled, y_scaled, deg=1)
line_mse = np.mean((y_scaled - (intercept + slope * x_scaled)) ** 2)
print("MSE of the straight line:", round(line_mse, 4))

plt.figure(figsize=(10, 5))
plt.scatter(x_scaled, y_scaled, alpha=0.5, s=20, label='Sampled data')
plt.plot(x_scaled, (g_func(x_raw) - np.mean(y_raw)) / np.std(y_raw), 'r-', linewidth=2, label='True function')
plt.plot(x_scaled, intercept + slope * x_scaled, 'k--', linewidth=2, label='Linear regression')
plt.xlabel('x (scaled)')
plt.ylabel('y (scaled)')
plt.legend()
plt.grid(True)
plt.show()

## Common Activation Functions

$$
\text{ReLU}(z) = \max(0, z)
$$

$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

$$
\tanh(z) = \frac{e^{z} - e^{-z}}{e^{z} + e^{-z}}
$$

## The Model

## The Model in Matrix Notation

$$
h = x W_h + b_h = x \begin{bmatrix}
w_{h1} & w_{h2}
\end{bmatrix} + \begin{bmatrix}
b_{h1} & b_{h2}
\end{bmatrix} = \begin{bmatrix}
w_{h1} x + b_{h1} & w_{h2} x + b_{h2}
\end{bmatrix}
$$

$$
z = \text{ReLU}(h) = \begin{bmatrix}
\text{ReLU}(w_{h1} x + b_{h1}) & \text{ReLU}(w_{h2} x + b_{h2})
\end{bmatrix}
$$

$$
\hat{Y} = Z W_o + b_o
$$

## Weights Update Equations

### Output Layer Weights and Bias

$$
\underbrace{\frac{\partial L}{\partial W_o}}_{2 \times 1} = -\frac{2}{n} 
\begin{bmatrix}
\sum_{i=1}^{n} (y_i - \hat{y}_i) z_{i1} \\
\sum_{i=1}^{n} (y_i - \hat{y}_i) z_{i2}
\end{bmatrix} =
-\frac{2}{n} \underbrace{Z^\top}_{2 \times n} \underbrace{(y - \hat{y})}_{n \times 1}
$$

$$
\frac{\partial L}{\partial b_o} = -\frac{2}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)
$$

### Hidden Layer Weights and Biases

$$
\text{ReLU}'(z) = \begin{cases}
1 & \text{if } z > 0 \\
0 & \text{otherwise}
\end{cases}
$$

$$
\underbrace{\Delta}_{n \times k} = \left( (y - \hat{y}) W_o^\top \right) \odot \text{ReLU}'(H)
$$

$$
\underbrace{\frac{\partial L}{\partial W_h}}_{1 \times k} = -\frac{2}{n} \underbrace{x^\top}_{1 \times n} \Delta
\qquad
\underbrace{\frac{\partial L}{\partial b_h}}_{1 \times k} = -\frac{2}{n} \underbrace{\mathbf{1}^\top}_{1 \times n} \Delta
$$

## Tanh Activation

## Implementation {#implementation}

### The worked example in NumPy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.array([[4], [2]])
print("Input matrix shape:", x.shape)
print(x)

W_h = np.array([[0.25, 0.5]]).reshape(-1, 2)
print("Hidden layer weights shape:", W_h.shape)
print(W_h)

b_h = np.array([0.1, -1.1]).reshape(-1, 2)
print("Hidden layer biases shape:", b_h.shape)
print(b_h)

# Compute hidden layer net values (pre-activations)

H = x @ W_h + b_h
print("Hidden layer shape:", H.shape)
print(H)

def relu(x):
    return np.maximum(0, x)

Z = relu(H)
print("Activated hidden layer shape:", Z.shape)
print(Z)

W_o = np.array([0.4, -0.6]).reshape(-1, 1)

print("Output layer weights shape:", W_o.shape)
print(W_o)

b_o = np.array([-0.2]).reshape(-1, 1)
print("Output layer biases shape:", b_o.shape)
print(b_o)

y_pred = Z @ W_o + b_o
print("Output shape:", y_pred.shape)
print(y_pred)

y_example = np.array([[1.0], [0.0]])
print("Target shape:", y_example.shape)
print(y_example)

np.mean((y_example - y_pred)**2)

### Training with NumPy

In [ ]:
# Neural network regression using NumPy with ReLU activation

# First we set the random seed for reproducibility
# as it is used to initialize the weights

np.random.seed(42)

# Number of hidden units (the example above used 2, here we use 3)
k = 3

# Initialize the weights and biases (He initialization for ReLU: variance 2 / fan_in)
W_h = np.random.randn(1, k) * np.sqrt(2.0 / 1)
b_h = np.zeros((1, k))
W_o = np.random.randn(k, 1) * np.sqrt(2.0 / k)
b_o = np.zeros((1, 1))

# Sanity check: print the shapes of the weights and biases
print("W_h shape:", W_h.shape)
print("b_h shape:", b_h.shape)
print("W_o shape:", W_o.shape)
print("b_o shape:", b_o.shape)

print("X shape:", X.shape)
print("y shape:", y.shape)

# Some helper function to make the code
# below easier to read

def relu(x):
    return np.maximum(0, x)

def relu_prime(x):
    return (x > 0).astype(float)

# Training parameters: plain full-batch gradient descent
epochs = 1000
learning_rate = 0.03
losses_np = []

# Training loop
for epoch in range(epochs):
    # Forward pass (calculate predictions)

    H = X @ W_h + b_h
    Z = relu(H)
    y_pred = Z @ W_o + b_o
    
    # Compute loss (MSE)
    loss = np.mean((y - y_pred) ** 2)
    losses_np.append(loss)
    
    # Backward pass
    dLoss_dy = -2 * (y - y_pred) / n
    
    # Output layer gradients
    dW_o = Z.T @ dLoss_dy
    db_o = np.sum(dLoss_dy)
    
    # Hidden layer gradients
    dZ = dLoss_dy @ W_o.T
    dH = dZ * relu_prime(H) # Note the element-wise (not matrix) multiplication here
    dW_h = X.T @ dH
    db_h = np.sum(dH, axis=0, keepdims=True)
    
    # Update weights and biases
    W_o -= learning_rate * dW_o
    b_o -= learning_rate * db_o
    W_h -= learning_rate * dW_h
    b_h -= learning_rate * db_h

# Final predictions (after the training loop)
H = X @ W_h + b_h
Z = relu(H)

y_pred_final_np = Z @ W_o + b_o

# Plot results
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(X, y, alpha=0.5, s=20, label='Noisy data')
plt.plot(X, y_pred_final_np, 'r-', label='NumPy NN (ReLU)', linewidth=2)
plt.xlabel('x')
plt.ylabel('y')
plt.title('NumPy Neural Network with ReLU')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(losses_np)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss (NumPy)')
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Neural network regression using PyTorch

np.random.seed(42)
torch.manual_seed(42)

## Prepare the data
## torch has its own data structures called tensors
## and we need to convert the numpy arrays to tensors
## 
## Furthermore, the input to the network should be
## a 2D array (matrix) even if there is only one feature
## so the input shape should be (n, n_features)
## The output should also be a 2D array even if there is only one target
## 

x_train = torch.from_numpy(X).float()
y_train = torch.from_numpy(y).float()

print("x_train shape:", x_train.shape)
print("y_train shape:", y_train.shape)

################################
## This is the important part
################################

# Define the neural network
class TorchNN(nn.Module):
    def __init__(self, k=3):
        super().__init__()
        self.k = k

        self.input_to_hidden = nn.Linear(1, k)
        self.hidden_activation = nn.ReLU()
        self.output_layer = nn.Linear(k, 1)
    
    # Compute the forward pass (prediction)    
    def forward(self, x):
        # Map the input to hidden layer
        x = self.input_to_hidden(x)
        # Then apply activation to the result
        x = self.hidden_activation(x)

        # Then map the activated hidden layer to output
        x = self.output_layer(x)

        return x
    
##########################################
## The rest of the code runs the gradient
## descent algorithm to find the optimal 
## weights that minimize the MSE loss
##########################################

# Initialize model, loss, and optimizer
model = TorchNN(k=3)

criterion = nn.MSELoss()

# Note: Adam instead of the plain gradient descent used in the NumPy version
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Training loop

# How many times to iterate over the whole dataset 
# (one such iteration is called an epoch)

epochs = 1000
losses = []

for epoch in range(epochs):
    # Forward pass
    y_pred = model(x_train)
    loss = criterion(y_pred, y_train)
    losses.append(loss.item())
    
    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# Get predictions
with torch.no_grad():
    y_pred_final = model(x_train)

# Plot results
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(x_train, y_train, alpha=0.5, s=20, label='Noisy data')
plt.plot(x_train, y_pred_final.numpy(), 'r-', label='NN prediction', linewidth=2)
plt.xlabel('x')
plt.ylabel('y')
plt.title('Neural Network Approximation of Noisy Function')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.grid(True)

plt.tight_layout()
plt.show()

## What's next